In [1]:
import pandas as pd
import numpy as np 

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv(r"australia\australia.csv", low_memory=False)

print("Original shape:", df.shape)

Original shape: (53833, 513)


# Data Cleaning

In [4]:

# INITIAL DATA QUALITY CHECK


print("ORIGINAL DATASET OVERVIEW")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")


print("BEFORE CLEANING - MISSING VALUES")

missing_before = df.isna().sum()
missing_before_pct = (missing_before / len(df)) * 100
cols_with_missing = missing_before[missing_before > 0].sort_values(ascending=False)
print(f"Columns with missing values: {len(cols_with_missing)}")
print(f"Total missing cells: {missing_before.sum():,}")
print("\nTop 15 columns with most missing values:")
for col, count in cols_with_missing.head(15).items():
    pct = (count / len(df)) * 100
    print(f"  {col:50} {count:7,} ({pct:5.1f}%)")

# Treat empty/whitespace-only strings as missing

print("TREATING WHITESPACE-ONLY CELLS AS MISSING")

df_cleaned = df.copy()
df_cleaned = df_cleaned.replace(r'^\s*$', pd.NA, regex=True)
print("All empty strings and whitespace converted to NA")

# Compare before and after

print("AFTER CLEANING - MISSING VALUES")

missing_after = df_cleaned.isna().sum()
missing_after_pct = (missing_after / len(df_cleaned)) * 100
cols_with_missing_after = missing_after[missing_after > 0].sort_values(ascending=False)

print(f"Columns with missing values: {len(cols_with_missing_after)}")
print(f"Total missing cells: {missing_after.sum():,}")
print(f"Change: {missing_after.sum() - missing_before.sum():+,} cells newly treated as missing")

print("\nTop 15 columns with most missing values (after cleaning):")
for col, count in cols_with_missing_after.head(15).items():
    pct = (count / len(df_cleaned)) * 100
    before_count = df[col].isna().sum()
    newly_missing = count - before_count
    print(f"  {col:50} {count:7,} ({pct:5.1f}%) [+{newly_missing:,} new]")


print("SUMMARY")

print(f"Original missing cells:  {missing_before.sum():,}")
print(f"After whitespace clean:  {missing_after.sum():,}")
print(f"Additional missing found: {missing_after.sum() - missing_before.sum():,}")
print(f"Ready to proceed with column definition and analysis")

# Use cleaned dataframe going forward
df = df_cleaned


ORIGINAL DATASET OVERVIEW
Dataset shape: 53,833 rows × 513 columns
BEFORE CLEANING - MISSING VALUES
Columns with missing values: 111
Total missing cells: 4,368,582

Top 15 columns with most missing values:
  future_3                                            52,827 ( 98.1%)
  long_covid                                          50,004 ( 92.9%)
  had_covid_2                                         50,004 ( 92.9%)
  household_children_resp                             48,997 ( 91.0%)
  future_2                                            48,997 ( 91.0%)
  future_1                                            48,997 ( 91.0%)
  vac_boost_beyond                                    48,997 ( 91.0%)
  had_covid                                           48,997 ( 91.0%)
  vac12_booster_other                                 46,005 ( 85.5%)
  vac_boost_1                                         45,998 ( 85.4%)
  vac12_booster_10                                    45,998 ( 85.4%)
  vac12_booster_1       

In [5]:

# 1: Define important columns for my analysis


# Basic ID / time / demographic / weight columns
base_cols = [
    "RecordNo",
    "endtime",
    "qweek",
    "age",
    "gender",
    "state",
    "household_size",
    "employment_status",
    "weight"
]

# Main behaviour/adherence variables with minimal missing
main_behaviour_cols = [
    "i12_health_1",
    "i12_health_3",
    "i12_health_4",
    "i12_health_5",
    "i12_health_6",
    "i12_health_7",
    "i12_health_8",
    "i12_health_12",
    "i12_health_13",
    "i12_health_14",
    "i12_health_15",
    "i12_health_16"
]

# Behaviour variables with some missingness
mid_behaviour_cols = [
    "i12_health_2",
    "i12_health_11"
]

# Extra predictor variables
extra_candidate_cols = [
    "i9_health",
    "i11_health",
    "i2_health",
    "WCRex2",
    "cantril_ladder",
    "WCRex1"
]

# Combine all behaviour columns
all_behaviour_cols = main_behaviour_cols + mid_behaviour_cols

# All columns needed for analysis
selected_cols = base_cols + all_behaviour_cols + extra_candidate_cols


print("ANALYSIS VARIABLES DEFINED")

print(f"Protective behaviour variables: {len(all_behaviour_cols)}")
print(f"Base variables: {len(base_cols)}")
print(f"Predictor variables: {len(extra_candidate_cols)}")
print(f"Total variables for analysis: {len(selected_cols)}")


# 2: Data Quality Assessment for Analysis Variables



print("\nDATA COMPLETENESS FOR ANALYSIS VARIABLES")

print("\nMissing values in selected variables:")


missing_selected = df[selected_cols].isna().sum().sort_values(ascending=False)

# Separate into groups for clarity
behaviour_cols_check = [col for col in all_behaviour_cols if col in selected_cols]
base_cols_check = [col for col in base_cols if col in selected_cols]
extra_cols_check = [col for col in extra_candidate_cols if col in selected_cols]

print("\n1. PROTECTIVE BEHAVIOUR VARIABLES:")
for col in behaviour_cols_check:
    missing_count = df[col].isna().sum()
    status = "No Missing Values" if missing_count == 0 else f"  {missing_count:,}"
    print(f"   {col:45} {status}")

print("\n2. DEMOGRAPHIC & BASE VARIABLES:")
for col in base_cols_check:
    missing_count = df[col].isna().sum()
    status = "No Missing Values" if missing_count == 0 else f"  {missing_count:,}"
    print(f"   {col:45} {status}")

print("\n3. PREDICTOR VARIABLES:")
for col in extra_cols_check:
    missing_count = df[col].isna().sum()
    status = "No Missing Values" if missing_count == 0 else f"  {missing_count:,}"
    print(f"   {col:45} {status}")

# Summary

print("SUMMARY FOR SELECTED VARIABLES")

complete_vars = (missing_selected == 0).sum()
incomplete_vars = (missing_selected > 0).sum()
print(f"Complete variables (0 missing):      {complete_vars}")
print(f"Incomplete variables (with missing): {incomplete_vars}")
print(f"\nConclusion: {complete_vars}/{len(selected_cols)} selected variables are complete")
print(f"            {incomplete_vars} variables will require imputation or handling")


ANALYSIS VARIABLES DEFINED
Protective behaviour variables: 14
Base variables: 9
Predictor variables: 6
Total variables for analysis: 29

DATA COMPLETENESS FOR ANALYSIS VARIABLES

Missing values in selected variables:

1. PROTECTIVE BEHAVIOUR VARIABLES:
   i12_health_1                                  No Missing Values
   i12_health_3                                  No Missing Values
   i12_health_4                                  No Missing Values
   i12_health_5                                  No Missing Values
   i12_health_6                                  No Missing Values
   i12_health_7                                  No Missing Values
   i12_health_8                                  No Missing Values
   i12_health_12                                 No Missing Values
   i12_health_13                                 No Missing Values
   i12_health_14                                 No Missing Values
   i12_health_15                                 No Missing Values
   i12_hea

In [6]:

# 3: Inspect All Selected Columns - Value Types & Distribution



print("COMPREHENSIVE INSPECTION OF ALL 29 ANALYSIS VARIABLES")


# Create a summary table for all selected columns
inspection_summary = []

for col in selected_cols:
    dtype = df[col].dtype
    missing = df[col].isna().sum()
    unique_count = df[col].nunique()
    
    # Get sample values
    sample_values = df[col].dropna().unique()[:5]
    
    inspection_summary.append({
        'Column': col,
        'Data Type': dtype,
        'Missing': missing,
        'Unique': unique_count,
        'Sample Values': str(list(sample_values)[:3])
    })

inspection_df = pd.DataFrame(inspection_summary)

print("\nSUMMARY TABLE OF ALL VARIABLES:")

for idx, row in inspection_df.iterrows():
    print(f"{idx+1:2d}. {row['Column']:45} | Type: {str(row['Data Type']):10} | Missing: {row['Missing']:5} | Unique: {row['Unique']:4}")

# Detailed inspection by category

print("DETAILED VALUE INSPECTION BY CATEGORY")


categories = {
    "PROTECTIVE BEHAVIOUR VARIABLES": all_behaviour_cols,
    "DEMOGRAPHIC VARIABLES": base_cols,
    "PREDICTOR VARIABLES": extra_candidate_cols
}

for category_name, cols_list in categories.items():
    print(f"\n{category_name}:")
    for col in cols_list:
        if col in selected_cols:
            unique_vals = df[col].dropna().unique()
            print(f"\n  {col}:")
            print(f"    Data type: {df[col].dtype} | Missing: {df[col].isna().sum()} | Unique values: {len(unique_vals)}")
            
            # Show value counts
            if len(unique_vals) <= 10:
                print(f"    Values: {unique_vals}")
            else:
                print(f"    First 10 values: {unique_vals[:10]}")
                print(f"    ... and {len(unique_vals) - 10} more")

# Identify columns needing mapping

print("NEXT STEP: VALUE MAPPING STRATEGY")

print("\nIdentify which columns need numerical encoding:")
print("\nBEHAVIOUR VARIABLES (should have: Always/Frequently/Sometimes/Rarely/Not at all):")
print("  - Need frequency scale mapping: 5=Always, 4=Frequently, 3=Sometimes, 2=Rarely, 1=Not at all")

print("\nYES/NO VARIABLES (should have: Yes/No):")
print("  - Need binary mapping: 1=Yes, 0=No")

print("\nWILLINGNESS VARIABLES (should have: Very willing/Somewhat/Unwilling scale):")
print("  - Need ordinal mapping: 5=Very willing ... 1=Very unwilling")

print("\nPREDICTOR VARIABLES (check actual values):")
print("  - May need custom mapping based on response options")

print("\nNUMERIC VARIABLES (should be: household_size, age, etc.):")
print("  - May need type conversion or cleaning")


COMPREHENSIVE INSPECTION OF ALL 29 ANALYSIS VARIABLES

SUMMARY TABLE OF ALL VARIABLES:
 1. RecordNo                                      | Type: int64      | Missing:     0 | Unique: 53833
 2. endtime                                       | Type: object     | Missing:     0 | Unique: 44594
 3. qweek                                         | Type: object     | Missing:     0 | Unique:   54
 4. age                                           | Type: int64      | Missing:     0 | Unique:   79
 5. gender                                        | Type: object     | Missing:     0 | Unique:    2
 6. state                                         | Type: object     | Missing:     0 | Unique:    8
 7. household_size                                | Type: object     | Missing:     0 | Unique:   10
 8. employment_status                             | Type: object     | Missing:     0 | Unique:   10
 9. weight                                        | Type: float64    | Missing:     0 | Unique: 6289
10

In [7]:

# 4. Create working dataset


df_work = df[selected_cols].copy()

print("Working dataset shape:", df_work.shape)


Working dataset shape: (53833, 29)


In [8]:

# 5. Rename important columns to usable names


rename_dict = {
    # Base columns
    "RecordNo": "record_id",
    "endtime": "response_time",
    "qweek": "survey_week",
    "age": "age_group",
    "gender": "gender",
    "state": "state",
    "household_size": "household_size",
    "employment_status": "employment_status",
    "weight": "survey_weight",

    # Extra candidate columns
    "i2_health": "physical_contact_outside_household_count",
    "i9_health": "would_self_isolate_if_symptomatic",
    "i11_health": "willingness_to_self_isolate_7days",
    "WCRex1": "government_handling_covid",
    "WCRex2": "confidence_in_health_system_covid_response",
    "WCRV_4": "fear_of_contracting_covid",
    "CORE_B2_4": "happiness_compared_with_2_weeks_ago",
    "cantril_ladder": "life_satisfaction_ladder",

    # Main behaviour variables from i12_health
    "i12_health_1": "wore_mask_outside_home",
    "i12_health_2": "washed_hands_with_soap_water",
    "i12_health_3": "used_hand_sanitiser",
    "i12_health_4": "covered_nose_mouth_when_sneezing_coughing",
    "i12_health_5": "avoided_contact_with_symptomatic_people",
    "i12_health_6": "avoided_going_out_in_general",
    "i12_health_7": "avoided_going_to_healthcare_settings",
    "i12_health_8": "avoided_public_transport",
    "i12_health_11": "avoided_having_guests_at_home",
    "i12_health_12": "avoided_small_social_gatherings",
    "i12_health_13": "avoided_medium_social_gatherings",
    "i12_health_14": "avoided_large_social_gatherings",
    "i12_health_15": "avoided_crowded_areas",
    "i12_health_16": "avoided_going_to_shops",
    "i12_health_22": "wore_mask_in_grocery_store",
    "i12_health_23": "wore_mask_in_clothing_shop",
    "i12_health_25": "wore_mask_on_public_transport",
    "i12_health_26": "avoided_attending_public_events",

    # Risk / belief items
    "r1_1": "belief_covid_is_dangerous_for_me",
    "r1_2": "belief_likely_to_get_covid_future",
    "r1_3": "belief_mask_protects_me",
    "r1_4": "belief_mask_protects_others",
    "r1_5": "belief_mask_not_possible_for_me",
    "r1_6": "belief_health_improvement_important",
    "r1_7": "belief_life_greatly_affected_by_covid",
    "r1_8": "belief_vaccine_protects_me",
    "r1_9": "belief_vaccine_protects_others",
    "r1_10": "belief_vaccine_not_possible_for_me"
}

df_work = df_work.rename(columns=rename_dict)

In [9]:
df_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53833 entries, 0 to 53832
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   record_id                                   53833 non-null  int64  
 1   response_time                               53833 non-null  object 
 2   survey_week                                 53833 non-null  object 
 3   age_group                                   53833 non-null  int64  
 4   gender                                      53833 non-null  object 
 5   state                                       53833 non-null  object 
 6   household_size                              53833 non-null  object 
 7   employment_status                           53833 non-null  object 
 8   survey_weight                               53833 non-null  float64
 9   wore_mask_outside_home                      53833 non-null  object 
 10  used_hand_

In [10]:
# 6. Inspect unique values in important columns
cols = ['household_size',
 'wore_mask_outside_home',
 'used_hand_sanitiser',
 'covered_nose_mouth_when_sneezing_coughing',
 'avoided_contact_with_symptomatic_people',
 'avoided_going_out_in_general',
 'avoided_going_to_healthcare_settings',
 'avoided_public_transport',
 'avoided_small_social_gatherings',
 'avoided_medium_social_gatherings',
 'avoided_large_social_gatherings',
 'avoided_crowded_areas',
 'avoided_going_to_shops',
 'washed_hands_with_soap_water',
 'avoided_having_guests_at_home',
 'would_self_isolate_if_symptomatic',
 'willingness_to_self_isolate_7days',
 'physical_contact_outside_household_count',
 'confidence_in_health_system_covid_response',
 'life_satisfaction_ladder',
 'government_handling_covid']

for col in cols:
    print("Column:", col)
    print("Unique values:")
    print(df_work[col].dropna().unique())

Column: household_size
Unique values:
['2' '1' '3' '4' '5' 'Prefer not to say' '8 or more' '6' "Don't know" '7']
Column: wore_mask_outside_home
Unique values:
['Rarely' 'Not at all' 'Always' 'Frequently' 'Sometimes']
Column: used_hand_sanitiser
Unique values:
['Frequently' 'Always' 'Sometimes' 'Not at all' 'Rarely']
Column: covered_nose_mouth_when_sneezing_coughing
Unique values:
['Always' 'Frequently' 'Sometimes' 'Rarely' 'Not at all']
Column: avoided_contact_with_symptomatic_people
Unique values:
['Always' 'Frequently' 'Rarely' 'Not at all' 'Sometimes']
Column: avoided_going_out_in_general
Unique values:
['Always' 'Sometimes' 'Frequently' 'Not at all' 'Rarely']
Column: avoided_going_to_healthcare_settings
Unique values:
['Always' 'Sometimes' 'Frequently' 'Not at all' 'Rarely']
Column: avoided_public_transport
Unique values:
['Always' 'Sometimes' 'Not at all' 'Frequently' 'Rarely']
Column: avoided_small_social_gatherings
Unique values:
['Always' 'Frequently' 'Sometimes' 'Rarely' 'Not 

In [11]:
# Replace common text missing values
missing_like = [
    "Don't know",
    "Prefer not to say",
    "Not sure"
    ]

df_work = df_work.replace(missing_like, pd.NA)

In [12]:
df_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53833 entries, 0 to 53832
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   record_id                                   53833 non-null  int64  
 1   response_time                               53833 non-null  object 
 2   survey_week                                 53833 non-null  object 
 3   age_group                                   53833 non-null  int64  
 4   gender                                      53833 non-null  object 
 5   state                                       53833 non-null  object 
 6   household_size                              52474 non-null  object 
 7   employment_status                           53833 non-null  object 
 8   survey_weight                               53833 non-null  float64
 9   wore_mask_outside_home                      53833 non-null  object 
 10  used_hand_

In [13]:
# 7. Convert response_time
# -----------------------------
df_work["response_time"] = pd.to_datetime(
    df_work["response_time"],
    dayfirst=True,
    errors="coerce")

In [14]:


# 8. Define mappings


# Always/Frequently/Sometimes/Rarely/Not at all
freq_map = {
    "Always": 5,
    "Frequently": 4,
    "Sometimes": 3,
    "Rarely": 2,
    "Not at all": 1
}

# Yes/No
yes_no_map = {
    "Yes": 1,
    "No": 0
}

# Willingness scale
willing_map = {
    "Very willing": 5,
    "Somewhat willing": 4,
    "Neither willing nor unwilling": 3,
    "Somewhat unwilling": 2,
    "Very unwilling": 1
}

# Government handling scale
gov_map = {
    "Very well": 5,
    "Somewhat well": 4,
    "Somewhat badly": 2,
    "Very badly": 1
}

# Confidence in health system scale
confidence_map = {
    "A lot of confidence": 4,
    "A fair amount of confidence": 3,
    "Not very much confidence": 2,
    "No confidence at all": 1
}


# 8.1 Columns to convert


# Behaviour columns with Always/Frequently/etc.
freq_cols = [
    "wore_mask_outside_home",
    "used_hand_sanitiser",
    "covered_nose_mouth_when_sneezing_coughing",
    "avoided_contact_with_symptomatic_people",
    "avoided_going_out_in_general",
    "avoided_going_to_healthcare_settings",
    "avoided_public_transport",
    "avoided_small_social_gatherings",
    "avoided_medium_social_gatherings",
    "avoided_large_social_gatherings",
    "avoided_crowded_areas",
    "avoided_going_to_shops",
    "washed_hands_with_soap_water",
    "avoided_having_guests_at_home"
]

# Yes/No columns
yes_no_cols = [
    "would_self_isolate_if_symptomatic"
]

# Willingness column
willing_cols = [
    "willingness_to_self_isolate_7days"
]


# 8.2 Apply mappings

# Frequency scale
for col in freq_cols:
    df_work[col] = df_work[col].map(freq_map).astype("Int64")

# Yes/No scale
for col in yes_no_cols:
    df_work[col] = df_work[col].map(yes_no_map).astype("Int64")

# Willingness scale
for col in willing_cols:
    df_work[col] = df_work[col].map(willing_map).astype("Int64")

# Government handling
df_work["government_handling_covid"] = df_work["government_handling_covid"].map(gov_map).astype("Int64")

# Confidence in health system
df_work["confidence_in_health_system_covid_response"] = (
    df_work["confidence_in_health_system_covid_response"].map(confidence_map).astype("Int64")
)

In [15]:
# Replace "8 or more" with 8
df_work["household_size"] = df_work["household_size"].replace("8 or more", 8)

# Convert the column to numeric
df_work["household_size"] = pd.to_numeric(df_work["household_size"], errors="coerce")
df_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53833 entries, 0 to 53832
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   record_id                                   53833 non-null  int64         
 1   response_time                               53833 non-null  datetime64[ns]
 2   survey_week                                 53833 non-null  object        
 3   age_group                                   53833 non-null  int64         
 4   gender                                      53833 non-null  object        
 5   state                                       53833 non-null  object        
 6   household_size                              52474 non-null  float64       
 7   employment_status                           53833 non-null  object        
 8   survey_weight                               53833 non-null  float64       
 9   wore_m

In [16]:
# Convert to numeric
numeric_col = [
    'household_size',
    'physical_contact_outside_household_count',
    'life_satisfaction_ladder'
]

for col in numeric_col:
    df_work[col] = df_work[col].astype("Int64")

In [17]:
# Convert to category
category_col = [
    'survey_week', 'gender','state','employment_status'
]

for col in category_col:
    df_work[col] = df_work[col].astype("category")

In [18]:
df_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53833 entries, 0 to 53832
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   record_id                                   53833 non-null  int64         
 1   response_time                               53833 non-null  datetime64[ns]
 2   survey_week                                 53833 non-null  category      
 3   age_group                                   53833 non-null  int64         
 4   gender                                      53833 non-null  category      
 5   state                                       53833 non-null  category      
 6   household_size                              52474 non-null  Int64         
 7   employment_status                           53833 non-null  category      
 8   survey_weight                               53833 non-null  float64       
 9   wore_m

In [55]:
null_values= df_work.isna().sum().sort_values(ascending=False)
null_values

government_handling_covid                     8888
would_self_isolate_if_symptomatic             5635
confidence_in_health_system_covid_response    4927
life_satisfaction_ladder                      3985
willingness_to_self_isolate_7days             2068
physical_contact_outside_household_count      2002
household_size                                1359
avoided_having_guests_at_home                 1006
washed_hands_with_soap_water                  1006
avoided_public_transport                         0
avoided_going_to_shops                           0
avoided_crowded_areas                            0
avoided_large_social_gatherings                  0
avoided_medium_social_gatherings                 0
avoided_small_social_gatherings                  0
record_id                                        0
response_time                                    0
avoided_going_out_in_general                     0
avoided_contact_with_symptomatic_people          0
covered_nose_mouth_when_sneezin

In [19]:

# 1. Make a copy before imputation

df_imputed = df_work.copy()


# 2. Median imputation columns

median_cols = [
    "washed_hands_with_soap_water",
    "avoided_having_guests_at_home",
    "household_size",
    "physical_contact_outside_household_count",
    "willingness_to_self_isolate_7days",
    "life_satisfaction_ladder",
    "confidence_in_health_system_covid_response",
    "government_handling_covid"
]

for col in median_cols:
    median_value = df_imputed[col].median()
    df_imputed[col] = df_imputed[col].fillna(median_value)


# 3. Mode imputation columns

mode_cols = [
    "would_self_isolate_if_symptomatic"
]

for col in mode_cols:
    mode_value = df_imputed[col].mode()[0]
    df_imputed[col] = df_imputed[col].fillna(mode_value)


# 4. Convert imputed numeric columns back to Int64 if needed

imputed_int_cols = median_cols + mode_cols

for col in imputed_int_cols:
    df_imputed[col] = df_imputed[col].round().astype("Int64")


In [20]:
df_imputed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53833 entries, 0 to 53832
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   record_id                                   53833 non-null  int64         
 1   response_time                               53833 non-null  datetime64[ns]
 2   survey_week                                 53833 non-null  category      
 3   age_group                                   53833 non-null  int64         
 4   gender                                      53833 non-null  category      
 5   state                                       53833 non-null  category      
 6   household_size                              53833 non-null  Int64         
 7   employment_status                           53833 non-null  category      
 8   survey_weight                               53833 non-null  float64       
 9   wore_m

In [21]:
# 7. Saving the YouGov data as csv
csv_name = 'UGov_Data.csv'
df_imputed.to_csv(csv_name, index=False)